# 01c. QuartzNet + CR-CTC — регуляризация консистентностью

## Идея

CR-CTC (Consistency Regularization CTC) вводит дополнительный loss на симметричное KL-расстояние
между предсказаниями двух аугментированных версий одного и того же аудио.
Это побуждает энкодер давать устойчивые предсказания при вариациях входа.

**Два view одного батча.** В идеале оба view генерируются в DataLoader.
Однако в Wave-1 параметр `return_two_views` удалён из `SpokenNumbersDataset`.

**Упрощение (для демонстрации):** второй view генерируется на лету в цикле:
батч уже содержит augmented audio (view 1 через Dataset), а view 2 — результат
применения второго аугментера к тому же сырому аудио. Для этого в тренировочном цикле
audio передаётся через `aug2` напрямую (поверх уже аугментированного Dataset-ом).

> Упрощение: второй view делается на лету из уже-аугментированного Dataset-ом audio,
> что не идеально (view2 — это «aug поверх aug», а не независимый вид оригинала).
> В production рекомендуется dataset с двумя view'ами.

**Адаптация под Wave-1 Batch:** поля `audio_view2` / `audio_view2_lengths` удалены из `Batch`.
View 2 строится в цикле через отдельный `AudioAugmenter`.

## Установка зависимостей

Выполнять под свою платформу — локально обычно уже установлено через `uv sync`; на Colab/Kaggle — раскомментируй нужный блок.

In [ ]:
# Локально: если репо уже установлен (uv sync / pip install -e .), эта ячейка — no-op.

# --- Colab ---
# !git clone --depth 1 https://github.com/DKazhekin/ITMO_Speech_Recognition_Course.git
# import sys
# sys.path.insert(0, "ITMO_Speech_Recognition_Course/group-projects/gp1/src")
# !pip install -q --upgrade num2words sentencepiece gdown

# --- Kaggle ---
# import sys
# sys.path.insert(0, "/kaggle/working/ITMO_Speech_Recognition_Course/group-projects/gp1/src")
# !pip install -q --upgrade num2words sentencepiece gdown

print("Deps cell — fill in your platform block above if on Colab/Kaggle.")

## Пути

`DATA_ROOT`, `CKPT_ROOT` и сопутствующие пути автоматически резолвятся в ячейке ниже относительно расположения ноутбука (`notebooks/experiments/..`). Правки требуются, только если структура репозитория изменена.

In [ ]:
# Idempotent data download: populates ../data/ with train/ dev/ test/ and
# their CSVs from the shared Google Drive ZIP. No-op if already present.
from pathlib import Path
import zipfile

import gdown

NOTEBOOK_DIR = Path.cwd()  # notebooks/experiments/
DATA_ROOT = (NOTEBOOK_DIR / ".." / "data").resolve()
ZIP_PATH = (NOTEBOOK_DIR / ".." / "data.zip").resolve()

if not DATA_ROOT.exists() or not any(DATA_ROOT.iterdir()):
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    if not ZIP_PATH.exists():
        gdown.download(
            url="https://drive.google.com/file/d/1WOubhQ4LtPYEZTOHNkZiDqIobfOQEWBW/view?usp=share_link",
            output=str(ZIP_PATH),
            quiet=False,
            fuzzy=True,
        )
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(DATA_ROOT.parent)
    print(f"Extracted to {DATA_ROOT}")
else:
    print(f"Data already present at {DATA_ROOT}")


In [ ]:
# Env hardening — MUST run before `import torch` in this process.
# PYTORCH_CUDA_ALLOC_CONF reduces VRAM fragmentation on torch.compile +
# variable T batches; cudnn.benchmark=False avoids autotune thrash under
# padded, variable-length inputs; matmul_precision="high" enables TF32.
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch

torch.backends.cudnn.benchmark = False
torch.set_float32_matmul_precision("high")


In [ ]:
# Paths — auto-resolved from DATA_ROOT defined in the download cell.
from pathlib import Path
import torch

TRAIN_ROOT = DATA_ROOT / "train"
DEV_ROOT = DATA_ROOT / "dev"
_TEST_DIR = DATA_ROOT / "test"
TEST_ROOT: Path | None = _TEST_DIR if _TEST_DIR.exists() else None
TRAIN_CSV = TRAIN_ROOT / "train.csv"
DEV_CSV = DEV_ROOT / "dev.csv"
CKPT_ROOT = (Path.cwd() / ".." / ".." / "checkpoints" / "01c_cr_ctc").resolve()

for p in (TRAIN_ROOT, DEV_ROOT, TRAIN_CSV, DEV_CSV):
    assert p.exists(), f"Path does not exist: {p}"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}, train={TRAIN_ROOT}, dev={DEV_ROOT}, ckpts={CKPT_ROOT}")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

from gp1.data.manifest import records_from_csv
from gp1.data.dataset import SpokenNumbersDataset
from gp1.data.collate import collate_fn
from gp1.data.audio_aug import AudioAugmenter
from gp1.data.spec_aug import SpecAugmenter
from gp1.features.melbanks import LogMelFilterBanks
from gp1.losses.ctc import CTCLoss
from gp1.models.quartznet import QuartzNet10x4
from gp1.train.optim import build_novograd
from gp1.train.schedulers import build_cosine_warmup
from gp1.train.checkpoint import save_best
from gp1.train.metrics import compute_cer, compute_per_speaker_cer
from gp1.decoding.greedy import greedy_decode
from gp1.text.vocab import CharVocab
from gp1.text.normalize import digits_to_words
from gp1.types import AugConfig

In [ ]:
# ---------------------------------------------------------------------------
# CRCTCLoss — скопировано из src/gp1/losses/cr_ctc.py (удалён в Wave 1)
# ---------------------------------------------------------------------------

class CRCTCLoss(nn.Module):
    """Consistency Regularization CTC Loss (CR-CTC).

    Computes symmetric KL divergence between two augmented views of the
    same utterance. Encourages the encoder to produce consistent predictions
    under different augmentations.

    Args:
        temperature: Softmax temperature for sharpening / smoothing (default 1.0).
        min_prob: Minimum confidence threshold; only frames where at least one
            view has max_prob >= min_prob contribute to the loss (default 0.1).
    """

    def __init__(self, temperature: float = 1.0, min_prob: float = 0.1) -> None:
        super().__init__()
        self.temperature = temperature
        self.min_prob = min_prob

    def forward(
        self,
        log_probs_a: torch.Tensor,     # [B, T, V] — view 1
        log_probs_b: torch.Tensor,     # [B, T, V] — view 2
        input_lengths: torch.Tensor,   # [B]
    ) -> torch.Tensor:
        assert log_probs_a.dim() == 3, "log_probs must be [B, T, V]"
        batch, time, _ = log_probs_a.shape

        if self.temperature != 1.0:
            log_probs_a = F.log_softmax(log_probs_a / self.temperature, dim=-1)
            log_probs_b = F.log_softmax(log_probs_b / self.temperature, dim=-1)

        probs_a = log_probs_a.exp()
        probs_b = log_probs_b.exp()

        lengths_mask = (
            torch.arange(time, device=log_probs_a.device)
            .unsqueeze(0)
            .lt(input_lengths.unsqueeze(1))
        )

        if self.min_prob > 0.0:
            conf_a = probs_a.max(dim=-1).values >= self.min_prob
            conf_b = probs_b.max(dim=-1).values >= self.min_prob
            conf_mask = conf_a | conf_b
        else:
            conf_mask = torch.ones(batch, time, dtype=torch.bool, device=log_probs_a.device)

        mask = lengths_mask & conf_mask
        n_valid = mask.sum().clamp(min=1)

        # Symmetric KL: 0.5 * (KL(a||b) + KL(b||a))
        kl_ab = (probs_a * (log_probs_a - log_probs_b)).sum(dim=-1)
        kl_ba = (probs_b * (log_probs_b - log_probs_a)).sum(dim=-1)
        sym_kl = 0.5 * (kl_ab + kl_ba)

        loss = (sym_kl * mask.float()).sum() / n_valid
        return loss

In [ ]:
# --- Manifests + vocab ---
train_records = records_from_csv(TRAIN_CSV, TRAIN_ROOT)
dev_records = records_from_csv(DEV_CSV, DEV_ROOT)
vocab = CharVocab()
print(f"Train: {len(train_records)} records. Dev: {len(dev_records)} records.")
print(f"Vocab size: {vocab.vocab_size}, blank_id: {vocab.blank_id}")

# In-domain / out-of-domain speaker split for harmonic CER metric.
in_domain_speakers = {r.spk_id for r in train_records}
out_of_domain_count = sum(1 for r in dev_records if r.spk_id not in in_domain_speakers)
in_domain_count = sum(1 for r in dev_records if r.spk_id in in_domain_speakers)
print(
    f"dev speakers: in-domain={in_domain_count} samples, "
    f"out-of-domain={out_of_domain_count} samples"
)


## Шаг 4.5. Предзагрузка аудио в RAM (опционально, сильно ускоряет обучение)

Загружает все train+dev файлы один раз, применяет resample до 16 kHz, и держит в RAM как тензоры. После этого `SpokenNumbersDataset.__getitem__` пропускает `sf.read` + `Resample` полностью.

Стоит ~3-5 минут один раз. Нужно ~2 GB RAM для GP1 (Colab: 12 GB, Kaggle: 29 GB — влезает с запасом).

In [ ]:
# FIXED — frontend / audio hyperparameters (not tunable in this experiment).
FIXED = {
    "samplerate": 16000,
    "n_fft": 512,
    "n_mels": 80,
    "hop_length": 160,
    "win_length": 400,
}
print("FIXED:", FIXED)


In [ ]:
# HP — tunable hyperparameters (single-trial values, no grid search).
HP = dict(
    lr=0.01,
    weight_decay=1e-3,
    dropout=0.1,
    d_model=256,
    subsample_factor=2,
    warmup_steps=3000,
    cr_ctc_weight=0.1,
    cr_temperature=1.0,
    cr_min_prob=0.1,
    max_epochs=30,
    grad_accum=2,
    batch_size=32,
    speed_prob=0.5,
    gain_prob=0.7,
    noise_prob=0.0,
    freq_mask_param=15,
    time_mask_param=25,
)
print("HP:", HP)


In [ ]:
from gp1.data.dataset import preload_audio_cache

AUDIO_CACHE = preload_audio_cache(
    train_records + dev_records,
    target_samplerate=FIXED["samplerate"],
)
# Move tensors to shared memory so DataLoader worker processes reuse the
# same underlying buffers instead of copying on every fork.
for k in list(AUDIO_CACHE.keys()):
    AUDIO_CACHE[k] = AUDIO_CACHE[k].contiguous().share_memory_()
print(f"AUDIO_CACHE: {len(AUDIO_CACHE)} entries")


In [ ]:
# DataLoader worker init — 1 BLAS thread per worker + per-worker RNG seed
# for the audio augmenter (avoids all workers sharing identical aug order).
import os
import random

import torch


def _worker_init(worker_id: int) -> None:
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    torch.set_num_threads(1)
    
    info = torch.utils.data.get_worker_info()
    if info is None:
        return
    aug = getattr(info.dataset, "_augmenter", None)
    if aug is not None and hasattr(aug, "_rng"):
        aug._rng = random.Random(info.seed & 0xFFFFFFFF)


BATCH_SIZE = 32
DL_WORKERS = 4

In [ ]:
aug_cfg1 = AugConfig(
    speed_prob=HP["speed_prob"],
    gain_prob=HP["gain_prob"],
    noise_prob=HP["noise_prob"],
    seed=42,
)
aug_cfg2 = AugConfig(
    speed_prob=HP["speed_prob"],
    gain_prob=HP["gain_prob"],
    noise_prob=HP["noise_prob"],
    seed=1337,
)
augmenter1 = AudioAugmenter(aug_cfg1)
augmenter2 = AudioAugmenter(aug_cfg2)  # applied manually in the training loop

train_ds = SpokenNumbersDataset(
    records=train_records,
    vocab=vocab,
    target_samplerate=FIXED["samplerate"],
    augmenter=augmenter1,
    audio_cache=AUDIO_CACHE,
)
dev_ds = SpokenNumbersDataset(
    records=dev_records,
    vocab=vocab,
    target_samplerate=FIXED["samplerate"],
    augmenter=None,
    audio_cache=AUDIO_CACHE,
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=DL_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    worker_init_fn=_worker_init,
)
dev_loader = DataLoader(
    dev_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=DL_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    worker_init_fn=_worker_init,
)
print(f"Train batches: {len(train_loader)}, Dev batches: {len(dev_loader)}")


## Развёрнутый тренировочный цикл — с CR-CTC

**Best practice — добавлен `torch.nn.utils.clip_grad_norm_`** с `max_norm=1.0` перед каждым шагом оптимизатора.
При CR-CTC loss добавляется второй forward pass модели, что увеличивает амплитуду градиентов.
Значение `max_norm=1.0` — стандартное для CTC-задач с NovoGrad.

In [ ]:
mel_extractor = LogMelFilterBanks(
    n_fft=FIXED["n_fft"], samplerate=FIXED["samplerate"],
    hop_length=FIXED["hop_length"], win_length=FIXED["win_length"],
    n_mels=FIXED["n_mels"],
).to(device)
spec_aug = SpecAugmenter(freq_mask_param=HP["freq_mask_param"], num_freq_masks=2,
                          time_mask_param=HP["time_mask_param"], num_time_masks=5, seed=42)
spec_aug2 = SpecAugmenter(freq_mask_param=HP["freq_mask_param"], num_freq_masks=2,
                           time_mask_param=HP["time_mask_param"], num_time_masks=5, seed=99)

model = QuartzNet10x4(
    vocab_size=vocab.vocab_size, d_model=HP["d_model"],
    dropout=HP["dropout"], subsample_factor=HP["subsample_factor"],
).to(device)

cr_loss_fn = CRCTCLoss(temperature=HP["cr_temperature"], min_prob=HP["cr_min_prob"])
ctc_loss_fn = CTCLoss(blank_id=vocab.blank_id)
all_params = list(model.parameters())
optimizer = build_novograd(all_params, lr=HP["lr"], betas=(0.95, 0.5),
                            weight_decay=HP["weight_decay"])
total_steps = len(train_loader) * HP["max_epochs"]
scheduler = build_cosine_warmup(optimizer, total_steps=total_steps,
                                 warmup_steps=HP["warmup_steps"])

ckpt_dir = CKPT_ROOT / "01c_cr_ctc"
best_cer = float("inf")
history = []
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
for epoch in tqdm(range(HP["max_epochs"]), desc="epochs"):
    model.train(); spec_aug.train(); spec_aug2.train()
    optimizer.zero_grad()
    grad_step = 0

    for step, batch in enumerate(tqdm(train_loader, desc=f"epoch {epoch}", leave=False)):
        audio = batch.audio.to(device)
        audio_lengths = batch.audio_lengths.to(device)
        targets = batch.targets.to(device)
        target_lengths = batch.target_lengths.to(device)

        # View 1: audio уже аугментирован Dataset'ом через augmenter1
        # View 2: применяем augmenter2 к тому же audio (уже на CPU side)
        # Упрощение: batch.audio — на device, нужно сгенерировать view2 с другим spec_aug
        with torch.no_grad():
            mel_v1 = mel_extractor(audio)

        mel_lengths = (
            (audio_lengths + FIXED["hop_length"] - 1) // FIXED["hop_length"]
        ).clamp(max=mel_v1.size(-1))

        # View 1: mel + spec_aug1
        mel_v1 = spec_aug(mel_v1, mel_lengths)

        # View 2: тот же mel, другой spec_aug (иной seed → другие маски)
        with torch.no_grad():
            mel_v2_base = mel_extractor(audio)
        mel_v2 = spec_aug2(mel_v2_base, mel_lengths)

        with torch.autocast(device_type=device.type, dtype=torch.float16,
                             enabled=(device.type == "cuda")):
            out_v1 = model(mel_v1, mel_lengths)
            out_v2 = model(mel_v2, mel_lengths)

        with torch.autocast(device_type=device.type, enabled=False):
            loss_main = ctc_loss_fn(
                out_v1.log_probs.float(), targets, out_v1.output_lengths, target_lengths
            )
            # --- CR-CTC: точка внедрения ---
            loss_cr = cr_loss_fn(
                out_v1.log_probs.float(), out_v2.log_probs.float(), out_v1.output_lengths
            )

        loss = loss_main + HP["cr_ctc_weight"] * loss_cr

        (loss / HP["grad_accum"]).backward()
        grad_step += 1
        if grad_step % HP["grad_accum"] == 0:
            torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()

    # Validation
    model.eval(); spec_aug.eval(); spec_aug2.eval()
    all_refs, all_hyps, all_spks = [], [], []

    with torch.no_grad():
        for batch in dev_loader:
            audio = batch.audio.to(device)
            audio_lengths = batch.audio_lengths.to(device)
            mel = mel_extractor(audio)
            mel_lengths = (
                (audio_lengths + FIXED["hop_length"] - 1) // FIXED["hop_length"]
            ).clamp(max=mel.size(-1))
            out = model(mel, mel_lengths)
            hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
            all_refs.extend(batch.transcriptions)
            all_hyps.extend(hyps)
            all_spks.extend(batch.spk_ids)

    ref_words = [digits_to_words(r) for r in all_refs]
    val_cer = compute_cer(ref_words, all_hyps)
    spk_cer = compute_per_speaker_cer(ref_words, all_hyps, all_spks)
    max_spk_cer = max(spk_cer.values()) if spk_cer else 0.0
    history.append({"epoch": epoch, "val_cer": val_cer, "max_spk_cer": max_spk_cer})
    tqdm.write(f"epoch {epoch:3d} | val_cer={val_cer:.4f} | max_spk_cer={max_spk_cer:.4f}")

    if val_cer < best_cer:
        best_cer = val_cer
        ckpt_path = save_best(
            model,
            meta=dict(arch="quartznet10x4", hparams=HP, val_cer=val_cer, epoch=epoch),
            ckpt_dir=ckpt_dir,
        )
        tqdm.write(f"  Checkpoint saved: {ckpt_path}")

## Итог

In [ ]:
import pandas as pd

df = pd.DataFrame(history)
print(f"Best val CER: {best_cer:.4f}")
print(f"Checkpoint dir: {ckpt_dir}")
print()
print(df.tail(10).to_string(index=False))

## TODO: harmonic CER integration

TODO: integrate harmonic in/out digit-CER into the manual loop's no-improve logic.

Current loop tracks `best_val_cer` via `compute_cer(ref_words, all_hyps)` against
`digits_to_words`-normalized references. The canonical block-L metric is
`compute_digit_cer_in_out_harmonic(refs_digits, hyps_digits, spk_ids,
in_domain_speakers)` applied after `safe_words_to_digits(h)` on hypotheses.
Wiring this into the manual loop requires replacing the `best_val_cer` check,
updating the `save_best` call's `val_cer` field, and extending `history` — left
as a follow-up to avoid invasive surgery on the manual training loop.
